# **Day 96: A GeoJSON Web API with Flask and SQLite**

Yesterday's app returned web pages for **people**. Today we build an **API** that returns **GeoJSON** for **programs**, like a web map (Leaflet, OpenLayers) or QGIS.

**The idea:** A database of **field survey points**. The API lets us:

| Method | URL | What it does |
|---|---|---|
| GET | `/api/points` | Get all points as GeoJSON (with optional filters) |
| GET | `/api/points/3` | Get one point |
| POST | `/api/points` | Add a new point |
| DELETE | `/api/points/3` | Delete a point |
| GET | `/api/stats` | Count of points by land cover |

This style of API is called a **REST API**.

In [1]:
%%writefile survey_api.py
import sqlite3
from flask import Flask, jsonify, request, g

DB = "survey.db"
app = Flask(__name__)
CLASSES = {"mangrove", "water", "agriculture", "settlement", "mudflat"}

def db():
    """Open one database connection per request."""
    if "db" not in g:
        g.db = sqlite3.connect(DB)
        g.db.row_factory = sqlite3.Row
    return g.db

@app.teardown_appcontext
def close_db(error):
    conn = g.pop("db", None)
    if conn:
        conn.close()

def init_db():
    with sqlite3.connect(DB) as c:
        c.executescript("""
        DROP TABLE IF EXISTS points;
        CREATE TABLE points (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            name TEXT NOT NULL,
            lat REAL NOT NULL CHECK (lat BETWEEN -90 AND 90),
            lon REAL NOT NULL CHECK (lon BETWEEN -180 AND 180),
            land_cover TEXT NOT NULL
        );
        INSERT INTO points (name, lat, lon, land_cover) VALUES
            ('SB001', 21.9452, 88.8931, 'mangrove'),
            ('SB002', 21.9610, 88.9105, 'water'),
            ('SB004', 22.0150, 88.8450, 'agriculture'),
            ('SB006', 21.9720, 88.9020, 'mudflat'),
            ('SB011', 21.9901, 88.8801, 'mangrove'),
            ('SB012', 22.0101, 88.8599, 'settlement');
        """)

def to_feature(row):
    """Turn one database row into a GeoJSON Feature."""
    return {"type": "Feature",
            "id": row["id"],
            "geometry": {"type": "Point", "coordinates": [row["lon"], row["lat"]]},
            "properties": {"name": row["name"], "land_cover": row["land_cover"]}}

@app.get("/api/points")
def list_points():
    sql = "SELECT * FROM points WHERE 1=1"
    args = []
    if cover := request.args.get("land_cover"):
        sql += " AND land_cover = ?"
        args.append(cover)
    if bbox := request.args.get("bbox"):              # bbox=min_lon,min_lat,max_lon,max_lat
        try:
            min_lon, min_lat, max_lon, max_lat = map(float, bbox.split(","))
        except ValueError:
            return jsonify(error="bbox must be 4 numbers: min_lon,min_lat,max_lon,max_lat"), 400
        sql += " AND lon BETWEEN ? AND ? AND lat BETWEEN ? AND ?"
        args += [min_lon, max_lon, min_lat, max_lat]
    rows = db().execute(sql, args).fetchall()
    return jsonify(type="FeatureCollection", features=[to_feature(r) for r in rows])

@app.get("/api/points/<int:pid>")
def get_point(pid):
    row = db().execute("SELECT * FROM points WHERE id = ?", (pid,)).fetchone()
    if row is None:
        return jsonify(error="not found"), 404
    return jsonify(to_feature(row))

@app.post("/api/points")
def add_point():
    data = request.get_json(silent=True)
    if not isinstance(data, dict):
        return jsonify(error="send a JSON object"), 400
    missing = {"name", "lat", "lon", "land_cover"} - data.keys()
    if missing:
        return jsonify(error=f"missing fields: {sorted(missing)}"), 400
    if data["land_cover"] not in CLASSES:
        return jsonify(error=f"land_cover must be one of {sorted(CLASSES)}"), 400
    try:
        cur = db().execute("INSERT INTO points (name, lat, lon, land_cover) VALUES (?, ?, ?, ?)",
                           (data["name"], float(data["lat"]), float(data["lon"]), data["land_cover"]))
        db().commit()
    except (sqlite3.IntegrityError, ValueError):
        return jsonify(error="lat/lon are not valid numbers or are out of range"), 400
    return jsonify(id=cur.lastrowid, **data), 201

@app.delete("/api/points/<int:pid>")
def delete_point(pid):
    cur = db().execute("DELETE FROM points WHERE id = ?", (pid,))
    db().commit()
    if cur.rowcount == 0:
        return jsonify(error="not found"), 404
    return "", 204

@app.get("/api/stats")
def stats():
    rows = db().execute("SELECT land_cover, COUNT(*) AS n FROM points GROUP BY land_cover ORDER BY n DESC").fetchall()
    return jsonify({r["land_cover"]: r["n"] for r in rows})

if __name__ == "__main__":
    init_db()
    app.run(debug=True)

Writing survey_api.py


## **Testing the API**

In [2]:
import json
from survey_api import app, init_db

init_db()
client = app.test_client()

def show(response):
    body = response.get_json(silent=True)
    if body is None:
        text = response.data.decode()
    else:
        text = json.dumps(body)
        if len(text) > 300:
            text = text[:300] + " ..."
    print(response.status_code, text)

show(client.get("/api/points"))

200 {"features": [{"geometry": {"coordinates": [88.8931, 21.9452], "type": "Point"}, "id": 1, "properties": {"land_cover": "mangrove", "name": "SB001"}, "type": "Feature"}, {"geometry": {"coordinates": [88.9105, 21.961], "type": "Point"}, "id": 2, "properties": {"land_cover": "water", "name": "SB002"},  ...


In [3]:
# Filter by land cover
r = client.get("/api/points?land_cover=mangrove")
print([f["properties"]["name"] for f in r.get_json()["features"]])

# Filter by a bounding box (only the points in this rectangle)
r = client.get("/api/points?bbox=88.85,21.95,88.92,22.00")
print([f["properties"]["name"] for f in r.get_json()["features"]])

['SB001', 'SB011']
['SB002', 'SB006', 'SB011']


In [4]:
show(client.get("/api/points/3"))
show(client.get("/api/points/99"))

200 {"geometry": {"coordinates": [88.845, 22.015], "type": "Point"}, "id": 3, "properties": {"land_cover": "agriculture", "name": "SB004"}, "type": "Feature"}
404 {"error": "not found"}


## **Adding and Deleting Points**

In [5]:
show(client.post("/api/points", json={"name": "SB013", "lat": 21.9800, "lon": 88.8700, "land_cover": "mangrove"}))
show(client.post("/api/points", json={"name": "SB014", "lat": 21.98}))                                  # Missing fields
show(client.post("/api/points", json={"name": "SB015", "lat": 91, "lon": 88.8, "land_cover": "water"}))   # Latitude 91 is impossible
show(client.post("/api/points", json={"name": "SB016", "lat": 21.9, "lon": 88.8, "land_cover": "forest?"}))

201 {"id": 7, "land_cover": "mangrove", "lat": 21.98, "lon": 88.87, "name": "SB013"}
400 {"error": "missing fields: ['land_cover', 'lon']"}
400 {"error": "lat/lon are not valid numbers or are out of range"}
400 {"error": "land_cover must be one of ['agriculture', 'mangrove', 'mudflat', 'settlement', 'water']"}


In [6]:
show(client.delete("/api/points/2"))
show(client.delete("/api/points/2"))       # Already deleted
show(client.get("/api/stats"))

204 
404 {"error": "not found"}
200 {"agriculture": 1, "mangrove": 3, "mudflat": 1, "settlement": 1}


## **Using the API from a Web Map**
Because the API returns standard GeoJSON, a web map can show the points directly. For example, with the **Leaflet** JavaScript library, one line is enough:
```javascript
fetch("/api/points").then(r => r.json()).then(data => L.geoJSON(data).addTo(map));
```
QGIS can also load it: **Layer -> Add Layer -> Add Vector Layer -> Protocol: HTTP(S)** and give the URL.

## **Good Practices We Followed**
- **Check every input.** Never trust data sent by a user.
- Use the right **status codes**: 200 OK, 201 Created, 204 No Content, 400 Bad Request, 404 Not Found.
- Always use `?` placeholders in SQL (Day 75).
- Return **standard formats** (GeoJSON), so any GIS tool can use the data.

In real projects, people often use **FastAPI** (it checks data types and creates documentation automatically) and **PostGIS** (a spatial database).

In [7]:
# Quick question
# Which HTTP method is used to add a new point?